# 🟡 Day 5 — Build a Chatbot with `MessagesState`

**LangGraph from Zero to Advanced · Day 5 of 16**

**Today's goal:** let LangGraph manage the chat history for you using the `add_messages` reducer and the ready-made `MessagesState`.

| Lesson | Topic |
|---|---|
| 1 | `add_messages` — the reducer behind every chatbot |
| 2 | `MessagesState` — ready-made chat state |
| 3 | Your first chatbot graph |
| 4 | System prompt inside the node |
| 5 | Multi-turn chat |
| 6 | Keep the history short (`trim_messages`) |
| 7 | Mini project: Study Buddy with modes |
| 8 | Practice |

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Lesson 1 — `add_messages`: the reducer behind every chatbot

On Day 2 you used `operator.add` to append to a list. For messages LangGraph has a smarter reducer, **`add_messages`**:

1. **Appends** new messages.
2. **Converts** shortcuts like `("user", "hi")` into real message objects.
3. Gives every message an **id**.
4. A message with an **existing id replaces** the old one.
5. `RemoveMessage(id=...)` **deletes** a message.

Let's call it directly to see each rule.

In [ ]:
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, RemoveMessage

# 1+2+3: append and convert
msgs = add_messages([], [("user", "Hello!"), ("assistant", "Hi there!")])
for m in msgs:
    print(m.type, "|", m.content, "| id:", m.id[:8])

In [ ]:
# 4: replace — same id overwrites
first_id = msgs[0].id
msgs = add_messages(msgs, [HumanMessage(content="Hello (edited)!", id=first_id)])
print([m.content for m in msgs])

# 5: remove by id
msgs = add_messages(msgs, [RemoveMessage(id=first_id)])
print([m.content for m in msgs])

## Lesson 2 — `MessagesState`

Instead of writing `messages: Annotated[list[AnyMessage], add_messages]` every time, use the built-in **`MessagesState`**. You can extend it with your own keys.

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph, START, END, MessagesState

# These two are the same thing:
class MyChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# MessagesState already contains `messages`. Add extra keys like this:
class ChatState(MessagesState):
    user_name: str

print(list(ChatState.__annotations__))

## Lesson 3 — Your first chatbot graph

```
START ──► chatbot ──► END
```

The node sends **all** messages to the LLM and returns **only the new reply** (in a list). `add_messages` appends it.

In [ ]:
def chatbot(state: MessagesState) -> dict:
    reply = llm.invoke(state["messages"])
    return {"messages": [reply]}          # ⚠️ a LIST with only the new message

g = StateGraph(MessagesState)
g.add_node("chatbot", chatbot)
g.add_edge(START, "chatbot")
g.add_edge("chatbot", END)
chat_app = g.compile()
show_graph(chat_app)

In [ ]:
result = chat_app.invoke({"messages": [("user", "Give me one fun fact about India.")]})
for m in result["messages"]:
    m.pretty_print()

assert len(result["messages"]) == 2      # 1 human + 1 AI
print("✅ Chatbot works!")

⚠️ **Most common beginner bug:** `return {"messages": state["messages"] + [reply]}`. Because of the reducer this **duplicates** the history. Always return only the new messages.

## Lesson 4 — System prompt inside the node

Add the system prompt **inside the node**, right before calling the LLM. Don't store it in the state — this keeps the history clean and lets you change the prompt any time.

In [ ]:
SYSTEM = SystemMessage(content="You are 'Chai', a friendly Python tutor. Keep answers under 60 words. Use one emoji.")

def tutor(state: MessagesState) -> dict:
    reply = llm.invoke([SYSTEM] + state["messages"])
    return {"messages": [reply]}

g = StateGraph(MessagesState)
g.add_node("tutor", tutor)
g.add_edge(START, "tutor")
g.add_edge("tutor", END)
tutor_app = g.compile()

out = tutor_app.invoke({"messages": [("user", "What is a list comprehension?")]})
print(out["messages"][-1].content)
assert all(m.type != "system" for m in out["messages"])   # system prompt NOT saved
print("✅ System prompt used but not stored")

## Lesson 5 — Multi-turn chat

Each `invoke()` starts fresh. For now we **carry the history ourselves** by passing the previous messages back in. (On Day 7 a *checkpointer* will do this automatically.)

In [ ]:
history = []
for user_text in ["Hi, I'm Rohan and I love cricket.", "What's my name and my hobby?"]:
    result = tutor_app.invoke({"messages": history + [HumanMessage(content=user_text)]})
    history = result["messages"]
    print("👤", user_text)
    print("🤖", history[-1].content, "\n")

print("Messages in history:", len(history))

### Chat with it yourself

Uncomment the last line and run. Type `quit` to stop.

In [ ]:
def chat():
    history = []
    while True:
        text = input("You: ")
        if text.lower() in ("quit", "exit", "q"):
            break
        history = tutor_app.invoke({"messages": history + [("user", text)]})["messages"]
        print("Bot:", history[-1].content)

# chat()

## Lesson 6 — Keep the history short

Long chats cost more tokens and may exceed the model's limit. `trim_messages` keeps only the **last N** messages (or last N tokens).

In [ ]:
from langchain_core.messages import trim_messages

long_chat = add_messages([], [
    ("user", "msg 1"), ("assistant", "reply 1"),
    ("user", "msg 2"), ("assistant", "reply 2"),
    ("user", "msg 3"), ("assistant", "reply 3"),
    ("user", "msg 4"),
])

short = trim_messages(
    long_chat,
    strategy="last",       # keep the most recent
    token_counter=len,     # count MESSAGES instead of tokens (simple!)
    max_tokens=3,          # keep at most 3 messages
    start_on="human",      # the kept history must start with a user message
)
print([m.content for m in short])

Use it inside the node — the LLM sees a short history, but the full history stays in the state:

In [ ]:
def short_memory_bot(state: MessagesState) -> dict:
    recent = trim_messages(state["messages"], strategy="last", token_counter=len,
                           max_tokens=6, start_on="human")
    return {"messages": [llm.invoke([SYSTEM] + recent)]}

g = StateGraph(MessagesState)
g.add_node("bot", short_memory_bot)
g.add_edge(START, "bot")
g.add_edge("bot", END)
short_bot = g.compile()
print(short_bot.invoke({"messages": long_chat})["messages"][-1].content)

## Lesson 7 — Mini project: Study Buddy with modes

Combine Day 3 (routing) + Day 5 (chat):

```
                     ┌► explain ─┐
START ► detect_mode ─┼► quiz ────┼► END
                     └► summary ─┘
```

In [ ]:
from typing import Literal

class StudyState(MessagesState):
    mode: str

PROMPTS = {
    "explain": "You are a patient teacher. Explain simply with one example. Max 80 words.",
    "quiz":    "Ask ONE multiple-choice question (A-D) about the topic. Don't reveal the answer.",
    "summary": "Summarize the topic in exactly 3 bullet points.",
}

def detect_mode(state: StudyState) -> dict:
    text = state["messages"][-1].content.lower()
    if "quiz" in text or "test me" in text:
        return {"mode": "quiz"}
    if "summar" in text or "tl;dr" in text:
        return {"mode": "summary"}
    return {"mode": "explain"}

def make_node(mode):
    def node(state: StudyState) -> dict:
        reply = llm.invoke([SystemMessage(content=PROMPTS[mode])] + state["messages"])
        return {"messages": [reply]}
    return node

def route_mode(state: StudyState) -> Literal["explain", "quiz", "summary"]:
    return state["mode"]

g = StateGraph(StudyState)
g.add_node("detect_mode", detect_mode)
for mode in PROMPTS:
    g.add_node(mode, make_node(mode))
    g.add_edge(mode, END)
g.add_edge(START, "detect_mode")
g.add_conditional_edges("detect_mode", route_mode)
study = g.compile()
show_graph(study)

In [ ]:
for q in ["What is recursion?", "Quiz me on Python dictionaries", "Summarize what LangGraph is"]:
    out = study.invoke({"messages": [("user", q)]})
    print(f"👤 {q}\n📘 mode={out['mode']}\n🤖 {out['messages'][-1].content}\n")

## Lesson 8 — Practice

**Exercise 1.** Build a chatbot whose state also has `turns: int`. Each run adds 1. Run it 3 times passing the history and turns back in.

**Exercise 2.** Write a node `forget_old` that deletes all but the last 2 messages using `RemoveMessage`. Put it **after** the chatbot node.

In [ ]:
# ✅ Solution 1
class CountingState(MessagesState):
    turns: int

def counting_bot(state: CountingState) -> dict:
    return {"messages": [llm.invoke(state["messages"])], "turns": state.get("turns", 0) + 1}

g = StateGraph(CountingState)
g.add_node("bot", counting_bot)
g.add_edge(START, "bot")
g.add_edge("bot", END)
counting = g.compile()

state = {"messages": [], "turns": 0}
for text in ["hi", "how are you?", "bye"]:
    state = counting.invoke({"messages": state["messages"] + [("user", text)], "turns": state["turns"]})
print("turns:", state["turns"], "| messages:", len(state["messages"]))
assert state["turns"] == 3 and len(state["messages"]) == 6

In [ ]:
# ✅ Solution 2
def forget_old(state: MessagesState) -> dict:
    return {"messages": [RemoveMessage(id=m.id) for m in state["messages"][:-2]]}

g = StateGraph(MessagesState)
g.add_node("chatbot", chatbot)
g.add_node("forget_old", forget_old)
g.add_edge(START, "chatbot")
g.add_edge("chatbot", "forget_old")
g.add_edge("forget_old", END)
res = g.compile().invoke({"messages": [("user", "a"), ("assistant", "b"), ("user", "c")]})
print([m.type for m in res["messages"]])
assert len(res["messages"]) == 2
print("✅ All exercises solved!")

## 🎯 Day 5 Recap

* `add_messages` appends, converts, assigns ids, replaces and removes messages.
* `MessagesState` = ready-made state with a `messages` key. Extend it for extra keys.
* A chat node returns **only the new reply**: `{"messages": [reply]}`.
* Put the system prompt inside the node; use `trim_messages` to control length.

**Tomorrow (Day 6):** give the LLM **tools** (calculator, search, APIs) and build your first **agent**.